# V023: does a discrete derivative term explain controller traces?

The frozen protocol compares proportional/integral (PI) against its derivative extension (PID) on the same 576 approved original-A-fit breaths. Constant and linear target references, pressure lags 0/1, and fitting-only ordinary least squares/RANSAC are declared before outcomes. All comparisons use identical eligible rows and chronological fitting cuts. The nested PI cases are new matched controls rather than a claim that the previous V020 rows are identical.

A PID case must explain at least 10% of sampled breaths to the fixed near-exact control threshold and improve at least 2 percentage points over its matched PI case. A pass would justify developing a separate input-only reconstruction, not validation promotion or submission. Known pressure appears only in this conditional training diagnostic. Original A-validation and test pressures are never read. The sample overlaps 72 previously used E-resampling validation IDs, as already disclosed; E is not a pristine holdout.

Source and tests are in controller_pid_components_v023.ipynb. No original Kaggle data, solution code, pretrained solutions or unrelated data are used.

In [1]:
from pathlib import Path
from datetime import datetime,timezone
import json,hashlib,time,platform,sys,os
import numpy as np
import pandas as pd

def _v023_sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()

def _v023_code(path):
    return '\n'.join(''.join(c['source']) for c in json.loads(Path(path).read_text())['cells'] if c['cell_type']=='code')

def run_controller_pid_v023(root='.'):
    P=Path(root).resolve();O=P/'outputs/controller_pid_feasibility_v023'
    O.mkdir(parents=True,exist_ok=True)
    if (O/'summary.json').exists() or (O/'training_only_equation_fits.csv').exists():
        raise FileExistsError('Preserve existing diagnostic outcomes')
    started=time.monotonic();planpath=P/'research/controller_pid_protocol_v023.json'
    assert _v023_sha(planpath)=='2c28eaf78fdcf94eed25b172bfef86c14168d81b4a73b086663f92dfbe7fba62'
    plan=json.loads(planpath.read_text())
    component=P/plan['component_file'];component_code=_v023_code(component)
    assert hashlib.sha256(component_code.encode()).hexdigest()==plan['component_code_sha256']
    assert _v023_sha(P/plan['preflight_file'])==plan['preflight_file_sha256']
    ns={};exec(compile(component_code,str(component),'exec'),ns)
    assert all(ns['controller_selfcheck_v023']().values())
    assert _v023_sha(P/'data/train.csv')==plan['approved_train_sha256']
    for file,key in [('sample_source','sample_source_sha256'),('original_A_split_file','A_split_file_sha256'),('source_document','source_document_sha256')]:
        assert _v023_sha(P/plan[file])==plan[key]
    prior=json.loads((P/plan['sample_source']).read_text())
    sample_ids=np.asarray(prior['selected_A_fit_breath_ids'],np.int32)
    with np.load(P/plan['original_A_split_file']) as z:fit_ids=z['A_fit'].copy();val_ids=z['A_validation'].copy()
    assert len(sample_ids)==len(np.unique(sample_ids))==576 and np.isin(sample_ids,fit_ids).all() and not np.isin(sample_ids,val_ids).any()
    meta=pd.read_csv(P/'data/train.csv',usecols=['breath_id','R','C'],dtype=dict(breath_id='int32',R='int8',C='int8')).iloc[::80].reset_index(drop=True)
    selected=np.flatnonzero(meta.breath_id.isin(sample_ids).to_numpy());selected_set=set(selected.tolist())
    assert len(selected)==576 and (meta.iloc[selected].groupby(['R','C']).size()==64).all()
    frame=pd.read_csv(P/'data/train.csv',usecols=['breath_id','R','C','time_step','u_in','u_out','pressure'],
        skiprows=lambda i:i>0 and (i-1)//80 not in selected_set,
        dtype=dict(breath_id='int32',R='int8',C='int8',time_step='float64',u_in='float64',u_out='int8',pressure='float64'))
    assert len(frame)==576*80 and np.isin(frame.breath_id.to_numpy(),fit_ids).all() and not np.isin(frame.breath_id.to_numpy(),val_ids).any()
    rows=[]
    for bid,b in frame.groupby('breath_id',sort=False):
        u=b.u_in.to_numpy();p=b.pressure.to_numpy();phase=b.u_out.to_numpy()
        reference_steps=np.arange(int(np.flatnonzero(phase==0).max())+1,dtype=float)
        for family_i,family in enumerate(plan['families']):
            for lag in plan['pressure_lags']:
                X,target,k,n=ns['controller_design_v023'](u,p,phase,lag,family)
                if len(k)<plan['minimum_eligible_points']:continue
                cut=max(plan['minimum_fitting_points'],int(plan['chronological_fit_fraction']*len(k)))
                if cut<X.shape[1]+2 or len(k)-cut<plan['minimum_held_control_points']:continue
                for estimator in plan['estimators']:
                    seed=plan['RANSAC_seed']+int(bid)*17+family_i*2+lag
                    selected_fit=ns['select_fitting_candidate_v023'](X,target,cut,estimator,seed,
                        plan['RANSAC_draws'],plan['maximum_condition'])
                    if selected_fit is None:continue
                    coef,condition=selected_fit
                    physical,params=ns['physical_coefficients_v023'](coef,family,reference_steps,plan['physical_coefficient_tolerance'])
                    residual=X@coef-target;held=np.abs(residual[cut:]);fiterr=np.abs(residual[:cut])
                    threshold=plan['near_exact_held_median_control_residual']
                    near=bool(physical and np.median(held)<threshold and np.mean(held<threshold)>=plan['minimum_held_control_inlier_fraction'])
                    rows.append(dict(breath_id=int(bid),R=int(b.R.iloc[0]),C=int(b.C.iloc[0]),family=family,lag=lag,
                        estimator=estimator,seed=seed,eligible_points=len(k),fit_points=cut,held_control_points=len(held),
                        condition=condition,physical=physical,**params,fit_median_abs_control_residual=float(np.median(fiterr)),
                        held_median_abs_control_residual=float(np.median(held)),held_mean_abs_control_residual=float(np.mean(held)),
                        held_control_inlier_fraction=float(np.mean(held<threshold)),near_exact=near,
                        coefficients=json.dumps(coef.tolist()),control_positions=json.dumps(k.tolist()),
                        pressure_positions=json.dumps(n.tolist()),held_abs_residuals=json.dumps(held.tolist())))
    records=pd.DataFrame(rows)
    if records.empty:raise RuntimeError('No eligible diagnostic fits; inspect without loosening thresholds')
    records_path=O/'training_only_equation_fits.csv';records.to_csv(records_path,index=False,float_format='%.17g')
    summary=[]
    for family in plan['families']:
        for lag in plan['pressure_lags']:
            for estimator in plan['estimators']:
                g=records[(records.family==family)&(records.lag==lag)&(records.estimator==estimator)]
                summary.append(dict(family=family,lag=lag,estimator=estimator,eligible_breaths=len(g),
                    physical_count=int(g.physical.sum()),median_held_control_residual=float(g.held_median_abs_control_residual.median()) if len(g) else None,
                    near_exact_count=int(g.near_exact.sum()),near_exact_fraction_of_576=float(g.near_exact.sum()/576)))
    index={(a['family'],a['lag'],a['estimator']):a for a in summary};comparisons=[]
    for q in summary:
        if not q['family'].startswith('pid'):continue
        control=index[('pi_'+q['family'].split('_',1)[1],q['lag'],q['estimator'])]
        gain=q['near_exact_fraction_of_576']-control['near_exact_fraction_of_576']
        comparisons.append(dict(family=q['family'],lag=q['lag'],estimator=q['estimator'],
            PID_near_exact_count=q['near_exact_count'],PI_near_exact_count=control['near_exact_count'],
            PID_fraction=q['near_exact_fraction_of_576'],fraction_gain=gain,
            eligible=bool(q['near_exact_fraction_of_576']>=plan['minimum_PID_near_exact_fraction_of_576'] and gain>=plan['minimum_PID_fraction_gain_over_matched_PI'])))
    report=dict(status='complete_training_only_diagnostic',protocol_sha256=_v023_sha(planpath),
        source_code_sha256=hashlib.sha256(_v023_code(P/'research/controller_pid_feasibility_v023.ipynb').encode()).hexdigest(),
        component_code_sha256=plan['component_code_sha256'],source_document_sha256=plan['source_document_sha256'],
        approved_train_sha256=plan['approved_train_sha256'],records_file=str(records_path.relative_to(P)),records_sha256=_v023_sha(records_path),
        sample_training_breaths=576,selected_A_fit_breath_ids=sample_ids.tolist(),same_training_sample_as_V015_and_V020=True,
        no_A_validation_pressure_parsed=True,no_test_data_reads=True,raw_float64=True,
        diagnostic_units=plan['diagnostic_units'],families=summary,PID_vs_matched_PI=comparisons,
        gate_passed=bool(any(c['eligible'] for c in comparisons)),pressure_predictor_created=False,
        recovery_coverage_estimated=False,submission_created=False,runtime_seconds=time.monotonic()-started,
        runtime=dict(python=sys.version,numpy=np.__version__,pandas=pd.__version__,platform=platform.platform(),device='CPU'),
        completed_utc=datetime.now(timezone.utc).isoformat(),
        interpretation='Known training-pressure, discrete per-step PI/PID control consistency only. Later control rows were excluded from coefficient selection, but their pressures are known diagnostic regressors. This is not unseen-pressure accuracy. Failure concerns only these reference/lag/derivative/clipping families.')
    (O/'summary.json').write_text(json.dumps(report,indent=2,allow_nan=False)+'\n')
    print(pd.DataFrame(summary).to_string(index=False))
    print('\nPID versus matched PI:')
    print(pd.DataFrame(comparisons).to_string(index=False))
    print('\nTraining-only gate:',report['gate_passed'],'CPU seconds:',round(report['runtime_seconds'],2))
    print('No input-only pressure predictor, pressure MAE, GPU fit or submission was produced.')
    return report

print('V023 is inert by default. Execute run_controller_pid_v023(project_root) only after source/protocol review.')


V023 is inert by default. Execute run_controller_pid_v023(project_root) only after source/protocol review.
      family  lag              estimator  eligible_breaths  physical_count  median_held_control_residual  near_exact_count  near_exact_fraction_of_576
 pi_constant    0 ordinary_least_squares               432             243                      0.057202                25                    0.043403
 pi_constant    0 training_only_RANSAC64               432             313                      0.002162                28                    0.048611
 pi_constant    1 ordinary_least_squares               423              64                      0.534432                 0                    0.000000
 pi_constant    1 training_only_RANSAC64               423              82                      0.417757                 0                    0.000000
   pi_linear    0 ordinary_least_squares               432             221                      0.031734                18                